# 1. MITMA: exploración y Bronze
Leemos los originales locales con DuckDB. Bronze son los archivos sin modificar y tablas de lectura: los campos CSV se conservan como texto, incluidos `NA` y campos vacíos. Solo añadimos `_nivel` y `_archivo` como metadatos.

Esquema y decisiones: [bronze.md](../docs/bronze.md).

In [19]:
from pathlib import Path
import duckdb
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'mitma'
LOCAL = ROOT / 'data' / 'processed'
LOCAL.mkdir(parents=True, exist_ok=True)
con = duckdb.connect(str(LOCAL / 'mitma.duckdb'))
con.execute("SET memory_limit='80%'")
con.execute('CREATE SCHEMA IF NOT EXISTS bronze')

def literal(texto):
    return "'" + str(texto).replace("'", "''") + "'"


## Tablas

In [20]:
for indicador in ['viajes', 'personas', 'pernoctaciones']:
    partes = []
    for nivel in ['distritos', 'municipios', 'GAU']:
        ruta = DATA / f'por-{nivel}' / indicador / 'diarios' / '*.csv.gz'
        partes.append(f"""SELECT * EXCLUDE(filename), '{nivel}' AS _nivel,
            replace(filename, {literal(ROOT.as_posix() + '/')}, '') AS _archivo
            FROM read_csv({literal(ruta.as_posix())}, delim='|', header=true, all_varchar=true,
                          filename=true)""")
    con.execute(f'CREATE OR REPLACE TABLE bronze.{indicador} AS ' + ' UNION ALL '.join(partes))
    print(indicador)
    display(con.sql(f'DESCRIBE bronze.{indicador}').df())
    display(con.sql(f'SELECT * FROM bronze.{indicador} LIMIT 5').df())


viajes


,column_name,column_type,null,key,default,extra
0,fecha,VARCHAR,YES,None,None,None
1,periodo,VARCHAR,YES,None,None,None
2,origen,VARCHAR,YES,None,None,None
3,destino,VARCHAR,YES,None,None,None
4,distancia,VARCHAR,YES,None,None,None
5,actividad_origen,VARCHAR,YES,None,None,None
6,actividad_destino,VARCHAR,YES,None,None,None
7,estudio_origen_posible,VARCHAR,YES,None,None,None
8,estudio_destino_posible,VARCHAR,YES,None,None,None
9,residencia,VARCHAR,YES,None,None,None


,fecha,periodo,origen,destino,distancia,actividad_origen,actividad_destino,estudio_origen_posible,estudio_destino_posible,residencia,renta,edad,sexo,viajes,viajes_km,_nivel,_archivo
0,20230101,00,01001,01001,2-10,casa,frecuente,no,no,01,10-15,NA,NA,2.922,12.806,distritos,data/mitma/por-distritos/viajes/diarios/202301...
1,20230101,00,01001,01001,2-10,frecuente,casa,no,no,01,10-15,NA,NA,5.202,22.098,distritos,data/mitma/por-distritos/viajes/diarios/202301...
2,20230101,01,01001,01001,2-10,no_frecuente,no_frecuente,no,no,31,>15,25-45,NA,4.648,20.805,distritos,data/mitma/por-distritos/viajes/diarios/202301...
3,20230101,02,01001,01001,2-10,casa,frecuente,no,no,01,10-15,NA,NA,2.28,9.213,distritos,data/mitma/por-distritos/viajes/diarios/202301...
4,20230101,03,01001,01001,2-10,frecuente,casa,no,no,01,10-15,NA,NA,2.922,11.72,distritos,data/mitma/por-distritos/viajes/diarios/202301...


personas


,column_name,column_type,null,key,default,extra
0,fecha,VARCHAR,YES,None,None,None
1,zona_pernoctacion,VARCHAR,YES,None,None,None
2,edad,VARCHAR,YES,None,None,None
3,sexo,VARCHAR,YES,None,None,None
4,numero_viajes,VARCHAR,YES,None,None,None
5,personas,VARCHAR,YES,None,None,None
6,_nivel,VARCHAR,YES,None,None,None
7,_archivo,VARCHAR,YES,None,None,None


,fecha,zona_pernoctacion,edad,sexo,numero_viajes,personas,_nivel,_archivo
0,20230101,01001,0-25,hombre,0,251.165,distritos,data/mitma/por-distritos/personas/diarios/2023...
1,20230101,01001,0-25,hombre,1,53.172,distritos,data/mitma/por-distritos/personas/diarios/2023...
2,20230101,01001,0-25,hombre,2,59.478,distritos,data/mitma/por-distritos/personas/diarios/2023...
3,20230101,01001,0-25,hombre,2+,96.707,distritos,data/mitma/por-distritos/personas/diarios/2023...
4,20230101,01001,0-25,mujer,0,201.435,distritos,data/mitma/por-distritos/personas/diarios/2023...


pernoctaciones


,column_name,column_type,null,key,default,extra
0,fecha,VARCHAR,YES,None,None,None
1,zona_residencia,VARCHAR,YES,None,None,None
2,zona_pernoctacion,VARCHAR,YES,None,None,None
3,personas,VARCHAR,YES,None,None,None
4,_nivel,VARCHAR,YES,None,None,None
5,_archivo,VARCHAR,YES,None,None,None


,fecha,zona_residencia,zona_pernoctacion,personas,_nivel,_archivo
0,20230101,01001,01001,2329.644,distritos,data/mitma/por-distritos/pernoctaciones/diario...
1,20230101,01001,01017_AM,16.682,distritos,data/mitma/por-distritos/pernoctaciones/diario...
2,20230101,01001,01028_AM,2.922,distritos,data/mitma/por-distritos/pernoctaciones/diario...
3,20230101,01001,01051,6.056,distritos,data/mitma/por-distritos/pernoctaciones/diario...
4,20230101,01001,01054_AM,5.897,distritos,data/mitma/por-distritos/pernoctaciones/diario...


## Distribución

In [21]:
for tabla in ['viajes', 'personas', 'pernoctaciones']:
    print(tabla)
    display(con.sql(f'SUMMARIZE bronze.{tabla}').df())

viajes


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,fecha,VARCHAR,20230101,20230101,1,None,None,None,None,None,30558279,0.0
1,periodo,VARCHAR,00,23,24,None,None,None,None,None,30558279,0.0
2,origen,VARCHAR,01001,externo,4420,None,None,None,None,None,30558279,0.0
3,destino,VARCHAR,01001,externo,4420,None,None,None,None,None,30558279,0.0
4,distancia,VARCHAR,0.5-2,>50,4,None,None,None,None,None,30558279,0.0
5,actividad_origen,VARCHAR,casa,trabajo_estudio,4,None,None,None,None,None,30558279,0.0
6,actividad_destino,VARCHAR,casa,trabajo_estudio,4,None,None,None,None,None,30558279,0.0
7,estudio_origen_posible,VARCHAR,no,si,2,None,None,None,None,None,30558279,0.0
8,estudio_destino_posible,VARCHAR,no,si,2,None,None,None,None,None,30558279,0.0
9,residencia,VARCHAR,01,52,53,None,None,None,None,None,30558279,0.0


personas


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,fecha,VARCHAR,20230101,20230101,1,None,None,None,None,None,265893,0.0
1,zona_pernoctacion,VARCHAR,01001,GAU Ávila,4331,None,None,None,None,None,265893,0.0
2,edad,VARCHAR,0-25,65-100,4,None,None,None,None,None,265893,0.0
3,sexo,VARCHAR,hombre,mujer,2,None,None,None,None,None,265893,0.0
4,numero_viajes,VARCHAR,0,2+,4,None,None,None,None,None,265893,0.0
5,personas,VARCHAR,10.0,9996.749,122699,None,None,None,None,None,265893,0.0
6,_nivel,VARCHAR,GAU,municipios,3,None,None,None,None,None,265893,0.0
7,_archivo,VARCHAR,data/mitma/por-GAU/personas/diarios/20230101_P...,data/mitma/por-municipios/personas/diarios/202...,3,None,None,None,None,None,265893,0.0


pernoctaciones


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,fecha,VARCHAR,20230101,20230101,1,None,None,None,None,None,1722371,0.0
1,zona_residencia,VARCHAR,01001,5200108,3822,None,None,None,None,None,1722371,0.0
2,zona_pernoctacion,VARCHAR,01001,GAU Ávila,4331,None,None,None,None,None,1722371,0.0
3,personas,VARCHAR,0.0,9994.200,86241,None,None,None,None,None,1722371,0.0
4,_nivel,VARCHAR,GAU,municipios,3,None,None,None,None,None,1722371,0.0
5,_archivo,VARCHAR,data/mitma/por-GAU/pernoctaciones/diarios/2023...,data/mitma/por-municipios/pernoctaciones/diari...,3,None,None,None,None,None,1722371,0.0


## Otros

In [22]:
for familia in ['nombres', 'poblacion']:
    partes = []
    for nivel, sufijo in [('distritos', 'distritos'), ('municipios', 'municipios'), ('GAU', 'gaus')]:
        ruta = DATA / 'zonificación' / f'zonificacion_{nivel}' / f'{familia}_{sufijo}.csv'
        partes.append(f"""SELECT *, '{nivel}' AS _nivel,
            {literal(ruta.relative_to(ROOT).as_posix())} AS _archivo
            FROM read_csv({literal(ruta.as_posix())}, delim='|',
                header={'false' if familia == 'poblacion' else 'true'},
                all_varchar=true)""")
    con.execute(f'CREATE OR REPLACE TABLE bronze.{familia}_zonas AS ' + ' UNION ALL BY NAME '.join(partes))

auxiliares = {
    'correspondencias': DATA / 'zonificación' / 'relacion_ine_zonificacionMitma.csv',
    'calidad_distritos': DATA / 'calidad' / 'diarios' / '*.csv',
}
for tabla, ruta in auxiliares.items():
    con.execute(f"""CREATE OR REPLACE TABLE bronze.{tabla} AS
        SELECT * EXCLUDE(filename),
        replace(filename, {literal(ROOT.as_posix() + '/')}, '') AS _archivo
        FROM read_csv({literal(ruta.as_posix())}, delim='|', header=true,
            all_varchar=true, filename=true)""")

for tabla in ['nombres_zonas', 'poblacion_zonas', 'correspondencias', 'calidad_distritos']:
    print(tabla)
    display(con.sql(f'SELECT * FROM bronze.{tabla} LIMIT 5').df())
    display(con.sql(f'SUMMARIZE bronze.{tabla}').df())


nombres_zonas


,ID,name,_nivel,_archivo,column0
0,01001,Alegría-Dulantzi,distritos,data/mitma/zonificación/zonificacion_distritos...,None
1,01002,Amurrio,distritos,data/mitma/zonificación/zonificacion_distritos...,None
2,01004_AM,Artziniega agregacion de municipios,distritos,data/mitma/zonificación/zonificacion_distritos...,None
3,01009_AM,Asparrena agregacion de municipios,distritos,data/mitma/zonificación/zonificacion_distritos...,None
4,01010,Ayala/Aiara,distritos,data/mitma/zonificación/zonificacion_distritos...,None


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,ID,VARCHAR,01001,GAU Ávila,4413,None,None,None,None,None,8496,0.00
1,name,VARCHAR,A Coruña,Úbeda,4085,None,None,None,None,None,8496,0.00
2,_nivel,VARCHAR,GAU,municipios,3,None,None,None,None,None,8496,0.00
3,_archivo,VARCHAR,data/mitma/zonificación/zonificacion_GAU/nombr...,data/mitma/zonificación/zonificacion_municipio...,3,None,None,None,None,None,8496,0.00
4,column0,VARCHAR,0,999,2680,None,None,None,None,None,8496,69.19


poblacion_zonas


,column0,column1,_nivel,_archivo
0,01001,2925.0,distritos,data/mitma/zonificación/zonificacion_distritos...
1,01002,10307.0,distritos,data/mitma/zonificación/zonificacion_distritos...
2,01004_AM,3005.0,distritos,data/mitma/zonificación/zonificacion_distritos...
3,01009_AM,4599.0,distritos,data/mitma/zonificación/zonificacion_distritos...
4,01010,2951.0,distritos,data/mitma/zonificación/zonificacion_distritos...


...

,seccion_ine,distrito_ine,municipio_ine,distrito_mitma,municipio_mitma,gau_mitma,_archivo
0,0100101001,0100101,01001,01001,01001,01001,data/mitma/zonificación/relacion_ine_zonificac...
1,0100101002,0100101,01001,01001,01001,01001,data/mitma/zonificación/relacion_ine_zonificac...
2,0100201001,0100201,01002,01002,01002,01002,data/mitma/zonificación/relacion_ine_zonificac...
3,0100201002,0100201,01002,01002,01002,01002,data/mitma/zonificación/relacion_ine_zonificac...
4,0100201003,0100201,01002,01002,01002,01002,data/mitma/zonificación/relacion_ine_zonificac...


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,seccion_ine,VARCHAR,0100101001,NA,36722,None,None,None,None,None,36382,0.0
1,distrito_ine,VARCHAR,0100101,NA,10001,None,None,None,None,None,36382,0.0
2,municipio_ine,VARCHAR,01001,NA,8055,None,None,None,None,None,36382,0.0
3,distrito_mitma,VARCHAR,01001,5200108,3890,None,None,None,None,None,36382,0.0
4,municipio_mitma,VARCHAR,01001,52001,2682,None,None,None,None,None,36382,0.0
5,gau_mitma,VARCHAR,01001,GAU Ávila,2150,None,None,None,None,None,36382,0.0
6,_archivo,VARCHAR,data/mitma/zonificación/relacion_ine_zonificac...,data/mitma/zonificación/relacion_ine_zonificac...,1,None,None,None,None,None,36382,0.0


calidad_distritos


,fecha,zona,_archivo
0,20230101,10032_AM,data/mitma/calidad/diarios/20230101_distritos_...
1,20230101,19086_AM,data/mitma/calidad/diarios/20230101_distritos_...
2,20230101,19103_AM,data/mitma/calidad/diarios/20230101_distritos_...
3,20230101,19156_AM,data/mitma/calidad/diarios/20230101_distritos_...
4,20230101,25158_AM,data/mitma/calidad/diarios/20230101_distritos_...


,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,fecha,VARCHAR,20230101,20230101,1,None,None,None,None,None,9,0.0
1,zona,VARCHAR,10032_AM,4801701,9,None,None,None,None,None,9,0.0
2,_archivo,VARCHAR,data/mitma/calidad/diarios/20230101_distritos_...,data/mitma/calidad/diarios/20230101_distritos_...,1,None,None,None,None,None,9,0.0


## Geografía
Cargamos polígonos y centroides con la extensión `spatial` de DuckDB.

In [23]:
con.execute('INSTALL spatial')
con.execute('LOAD spatial')
for tabla, sufijo_archivo in [('zonas_geograficas', ''), ('centroides', '_centroides')]:
    partes = []
    for nivel, sufijo in [('distritos', 'distritos'), ('municipios', 'municipios'), ('GAU', 'gaus')]:
        ruta = DATA / 'zonificación' / f'zonificacion_{nivel}' / f'zonificacion_{sufijo}{sufijo_archivo}.shp'
        partes.append(f"""SELECT *, '{nivel}' AS _nivel,
            {literal(ruta.relative_to(ROOT).as_posix())} AS _archivo
            FROM ST_Read({literal(ruta.as_posix())})""")
    con.execute(f'CREATE OR REPLACE TABLE bronze.{tabla} AS ' + ' UNION ALL '.join(partes))
    display(con.sql(f'SELECT _nivel, ST_CRS(geom) AS crs, count(*) filas FROM bronze.{tabla} GROUP BY ALL').df())


,_nivel,crs,filas
0,GAU,EPSG:3042,2203
1,distritos,EPSG:3042,3909
2,municipios,EPSG:3042,2735


,_nivel,crs,filas
0,distritos,EPSG:3042,3909
1,municipios,EPSG:3042,2735
2,GAU,EPSG:3042,2203


## Decisiones para Silver
Silver preparará los datos para analizarlos, sin cambiar el significado de los valores publicados. A partir de las muestras y los resúmenes anteriores, tomamos estas decisiones:

- **Dar a cada campo el tipo adecuado.** Todo se ha leído como texto, por eso `SUMMARIZE` no calcula medias ni cuartiles. Convertiremos `fecha` de `20230101` a `DATE`, `periodo` de `00`–`23` a entero y las medidas (`viajes`, `viajes_km`, `personas` y población) a `DOUBLE`. Las medidas tienen decimales en las muestras: no las redondearemos a enteros.
- **Mantener los códigos como texto.** Aparecen valores como `01001`, `01017_AM` y `GAU Ávila`. No son cantidades: convertirlos a números perdería ceros iniciales o impediría conservar las letras.
- **Tratar `NA` como dato no disponible.** Los resúmenes muestran `NA` en edad, sexo, población y algunos códigos INE, aunque el porcentaje de nulos sea cero: es texto, no un `NULL`. En Silver lo convertiremos a `NULL`, sin sustituirlo por cero ni eliminar la fila solo por ese motivo.
- **No mezclar niveles territoriales al sumar.** Las tablas reúnen distritos, municipios y GAU. Conservaremos `_nivel` para analizar cada nivel por separado y evitar contar varias veces la misma movilidad. Mantendremos también `_archivo` para saber de dónde procede cada fila.
- **Simplificar los catálogos.** En `poblacion_zonas`, renombraremos `column0` como `zona` y `column1` como `poblacion`. En `nombres_zonas`, eliminaremos `column0`, que es el índice original de municipios y no un atributo de la zona. No asignaremos un año a la población auxiliar porque estas celdas no muestran su fecha de referencia.

Fuente semántica: `data/mitma/LEEME_Especificaciones_indicadores_OpenDataMovilidad.xlsx`, pestañas EB de viajes, personas, pernoctaciones y distritos descartados.

In [24]:
con.close()
print('Bronze finished')

Bronze finished
